In [1]:
# @title 4D Lattice Yang-Mills Simulation (JAX)
# Copy this entire block into a code cell in Google Colab.
# Make sure to set Runtime -> Change runtime type -> T4 GPU (or better).
import subprocess
import sys
# Install JAX with CUDA support if not present (Colab usually has it, but good to ensure)
# subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "jax[cuda12]", "-f", "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html"])
import jax
import jax.numpy as jnp
from jax import random, jit, lax, vmap
import numpy as np
import time
print(f"JAX Version: {jax.__version__}")
print(f"Devices: {jax.devices()}")
# --- SU(2) Generators & Utils ---
sigma_x = jnp.array([[0, 1], [1, 0]], dtype=jnp.complex64)
sigma_y = jnp.array([[0, -1j], [1j, 0]], dtype=jnp.complex64)
sigma_z = jnp.array([[1, 0], [0, -1]], dtype=jnp.complex64)
sigmas = jnp.stack([sigma_x, sigma_y, sigma_z])
@jit
def exp_map_pauli(alpha):
    """Map R^3 vector alpha to SU(2) element U = exp(i * alpha . sigma)."""
    theta2 = jnp.sum(alpha**2)
    theta = jnp.sqrt(theta2 + 1e-10)
    c = jnp.cos(theta)
    s = jnp.sin(theta) / theta
    alpha_dot_sigma = jnp.einsum('k,kij->ij', alpha, sigmas)
    return c * jnp.eye(2, dtype=jnp.complex64) + 1j * s * alpha_dot_sigma
# --- Lattice Parameters ---
L = 8  # Lattice size (8^4 = 4096 sites) - Larger for Colab GPU
Dim = 4
Vol = L**Dim
NumLinks = Vol * Dim
# --- Neighbor Table (Precomputed on CPU) ---
def make_neighbor_table(L):
    sites = np.arange(L**4).reshape((L, L, L, L))
    up = []
    down = []
    for d in range(4):
        up.append(np.roll(sites, -1, axis=d).flatten())
        down.append(np.roll(sites, 1, axis=d).flatten())
    return jnp.array(up), jnp.array(down)
UP, DOWN = make_neighbor_table(L)
# --- Plaquette & Staples ---
@jit
def calculate_staples(U_flat, site, mu):
    """Calculate sum of staples around link (site, mu)."""
    staple_sum = jnp.zeros((2, 2), dtype=jnp.complex64)

    # Unrolled loop over directions nu != mu
    for nu in range(4):
        # Indices
        x = site
        x_plus_mu = UP[mu, x]
        x_plus_nu = UP[nu, x]
        x_minus_nu = DOWN[nu, x]
        x_plus_mu_minus_nu = DOWN[nu, x_plus_mu]

        # Forward Staple: U_nu(x+mu) U_mu(x+nu)^dag U_nu(x)^dag
        u1 = U_flat[x_plus_mu, nu]
        u2 = jnp.conjugate(jnp.transpose(U_flat[x_plus_nu, mu]))
        u3 = jnp.conjugate(jnp.transpose(U_flat[x, nu]))
        term_fwd = u1 @ u2 @ u3

        # Backward Staple: U_nu(x+mu-nu)^dag U_mu(x-nu)^dag U_nu(x-nu)
        u1_b = jnp.conjugate(jnp.transpose(U_flat[x_plus_mu_minus_nu, nu]))
        u2_b = jnp.conjugate(jnp.transpose(U_flat[x_minus_nu, mu]))
        u3_b = U_flat[x_minus_nu, nu]
        term_bwd = u3_b @ u2_b @ u1_b

        # Add to sum only if nu != mu
        mask = jnp.where(nu == mu, 0.0, 1.0)
        staple_sum += mask * (term_fwd + term_bwd)

    return staple_sum
@jit
def update_site_link(key, U_flat, site, mu, beta):
    """Metropolis update for a single link."""
    U = U_flat[site, mu]
    staples = calculate_staples(U_flat, site, mu)

    # Proposal
    key, subkey = random.split(key)
    alpha = random.normal(subkey, (3,)) * 0.1 # Step size
    dU = exp_map_pauli(alpha)
    U_new = dU @ U

    # Action change
    # S_loc = -beta/2 * ReTr(U * staples)
    S_old = -0.5 * beta * jnp.real(jnp.trace(U @ staples))
    S_new = -0.5 * beta * jnp.real(jnp.trace(U_new @ staples))
    dS = S_new - S_old

    # Accept/Reject
    accept = jnp.log(random.uniform(key)) < -dS
    U_final = jnp.where(accept, U_new, U)

    # Update lattice (functional update)
    U_flat = U_flat.at[site, mu].set(U_final)

    return key, U_flat, jnp.where(accept, 1, 0)
# --- Main Simulation ---
def run_simulation():
    print(f"Running Full Lattice Simulation on {L}^4 Lattice...")

    key = random.PRNGKey(42)
    # Cold start (Identity)
    U_flat = jnp.stack([jnp.eye(2, dtype=jnp.complex64) for _ in range(NumLinks)]).reshape((Vol, 4, 2, 2))

    beta = 5.0
    r_crit = 1.9248
    print(f"Beta: {beta}, r_crit: {r_crit}")

    # --- JIT Compiled Sweep ---
    @jit
    def sweep(key, U_flat):
        def body_fun(carry, idx):
            key, U_f = carry
            site = idx // 4
            mu = idx % 4
            key, U_f, acc = update_site_link(key, U_f, site, mu, beta)
            return (key, U_f), acc
        # Scan over all links
        (key, U_flat), accs = lax.scan(body_fun, (key, U_flat), jnp.arange(Vol * 4))
        return key, U_flat, jnp.sum(accs)
    # --- Thermalization ---
    print("Thermalizing...")
    start_time = time.time()
    for i in range(20): # 20 Sweeps
        key, U_flat, acc = sweep(key, U_flat)
        if i % 5 == 0:
            print(f"Sweep {i}, Acceptance: {acc / (Vol * 4):.2f}")
    print(f"Thermalization done in {time.time() - start_time:.2f}s")

    # --- Measurement ---
    print("Measuring Defects...")

    @jit
    def measure_defects(U_flat):
        # Radius r = arccos(0.5 * ReTr(U))
        def get_r(U):
            tr = jnp.real(jnp.trace(U))
            tr = jnp.clip(tr, -2.0, 2.0)
            return jnp.arccos(0.5 * tr)

        rs = vmap(vmap(get_r))(U_flat) # (Vol, 4)
        defects = rs > r_crit
        return defects, jnp.mean(defects)

    defects, rho = measure_defects(U_flat)
    print(f"Measured Defect Density: {rho:.4e}")

    n_defects = jnp.sum(defects)
    print(f"Total Defects: {n_defects} / {Vol * 4}")

    # Simple Clustering Check
    # We check if defects have defect neighbors.
    # This is expensive to code fully on GPU without graph libs,
    # so we just output the raw count for now.
    # If density is low (e.g. 1e-3), random collisions are rare.

    expected = (Vol * 4) * 1.9e-3
    print(f"Expected (Random Gas): {expected:.1f}")
if __name__ == "__main__":
    run_simulation()


JAX Version: 0.7.2
Devices: [CudaDevice(id=0)]
Running Full Lattice Simulation on 8^4 Lattice...
Beta: 5.0, r_crit: 1.9248
Thermalizing...
Sweep 0, Acceptance: 0.68
Sweep 5, Acceptance: 0.69
Sweep 10, Acceptance: 0.70
Sweep 15, Acceptance: 0.69
Thermalization done in 30.12s
Measuring Defects...
Measured Defect Density: 0.0000e+00
Total Defects: 0 / 16384
Expected (Random Gas): 31.1


In [2]:
# @title 4D Lattice YM Scaling Analysis (JAX)
# Copy this into a Colab cell (T4 GPU recommended).
import jax
import jax.numpy as jnp
from jax import random, jit, lax, vmap
import numpy as np
import time
import matplotlib.pyplot as plt
# --- Setup ---
L = 8
Dim = 4
Vol = L**Dim
NumLinks = Vol * Dim
sigma_x = jnp.array([[0, 1], [1, 0]], dtype=jnp.complex64)
sigma_y = jnp.array([[0, -1j], [1j, 0]], dtype=jnp.complex64)
sigma_z = jnp.array([[1, 0], [0, -1]], dtype=jnp.complex64)
sigmas = jnp.stack([sigma_x, sigma_y, sigma_z])
@jit
def exp_map_pauli(alpha):
    theta2 = jnp.sum(alpha**2)
    theta = jnp.sqrt(theta2 + 1e-10)
    c = jnp.cos(theta)
    s = jnp.sin(theta) / theta
    return c * jnp.eye(2, dtype=jnp.complex64) + 1j * s * jnp.einsum('k,kij->ij', alpha, sigmas)
# --- Neighbor Table ---
def make_neighbor_table(L):
    sites = np.arange(L**4).reshape((L, L, L, L))
    up = []
    down = []
    for d in range(4):
        up.append(np.roll(sites, -1, axis=d).flatten())
        down.append(np.roll(sites, 1, axis=d).flatten())
    return jnp.array(up), jnp.array(down)
UP, DOWN = make_neighbor_table(L)
# --- Update Step ---
@jit
def calculate_staples(U_flat, site, mu):
    staple_sum = jnp.zeros((2, 2), dtype=jnp.complex64)
    for nu in range(4):
        x_plus_mu = UP[mu, site]
        x_plus_nu = UP[nu, site]
        x_minus_nu = DOWN[nu, site]
        x_plus_mu_minus_nu = DOWN[nu, x_plus_mu]

        u1 = U_flat[x_plus_mu, nu]
        u2 = jnp.conjugate(jnp.transpose(U_flat[x_plus_nu, mu]))
        u3 = jnp.conjugate(jnp.transpose(U_flat[site, nu]))
        term_fwd = u1 @ u2 @ u3

        u1_b = jnp.conjugate(jnp.transpose(U_flat[x_plus_mu_minus_nu, nu]))
        u2_b = jnp.conjugate(jnp.transpose(U_flat[x_minus_nu, mu]))
        u3_b = U_flat[x_minus_nu, nu]
        term_bwd = u3_b @ u2_b @ u1_b

        mask = jnp.where(nu == mu, 0.0, 1.0)
        staple_sum += mask * (term_fwd + term_bwd)
    return staple_sum
@jit
def update_site_link(key, U_flat, site, mu, beta):
    U = U_flat[site, mu]
    staples = calculate_staples(U_flat, site, mu)
    key, subkey = random.split(key)
    alpha = random.normal(subkey, (3,)) * 0.1
    U_new = exp_map_pauli(alpha) @ U
    S_old = -0.5 * beta * jnp.real(jnp.trace(U @ staples))
    S_new = -0.5 * beta * jnp.real(jnp.trace(U_new @ staples))
    accept = jnp.log(random.uniform(key)) < -(S_new - S_old)
    return key, U_flat.at[site, mu].set(jnp.where(accept, U_new, U)), jnp.where(accept, 1, 0)
@jit
def sweep(key, U_flat, beta):
    def body(carry, idx):
        k, u = carry
        k, u, acc = update_site_link(k, u, idx // 4, idx % 4, beta)
        return (k, u), acc
    (key, U_flat), accs = lax.scan(body, (key, U_flat), jnp.arange(Vol * 4))
    return key, U_flat, jnp.sum(accs)
# --- Observables ---
@jit
def measure_plaquettes(U_flat):
    # Calculate average plaquette trace at each time slice t
    # We average over x, y, z and all 6 orientations
    # This is a simplified "Time-Slice Energy" correlator

    # Reshape to (L, L, L, L, 4, 2, 2)
    U = U_flat.reshape((L, L, L, L, 4, 2, 2))

    # We want P(t) = Sum_{x,y,z, mu<nu} ReTr(U_plaq(x,y,z,t))
    # This is expensive to code generically.
    # Let's just measure the "Polymer/Defect Density" profile P(t)
    # P(t) = Fraction of defects in slice t.

    def get_r(u):
        return jnp.arccos(jnp.clip(0.5*jnp.real(jnp.trace(u)), -1, 1))

    rs = vmap(vmap(vmap(vmap(vmap(get_r)))))(U) # (L,L,L,L,4)

    # Average over spatial volume L^3 and directions 4
    # P_t shape (L,)
    P_t = jnp.mean(rs, axis=(0, 1, 2, 4))
    return P_t
# --- Main Analysis ---
def run_scaling_analysis():
    betas = [2.5, 3.0, 3.5, 4.0, 4.5] # Range covering crossover to weak coupling
    results = {}

    key = random.PRNGKey(42)
    U_flat = jnp.stack([jnp.eye(2, dtype=jnp.complex64) for _ in range(NumLinks)]).reshape((Vol, 4, 2, 2))

    print(f"{'Beta':<10} | {'Accept':<10} | {'Mass Gap (est)':<15}")
    print("-" * 45)

    for beta in betas:
        # Thermalize
        for _ in range(20):
            key, U_flat, acc = sweep(key, U_flat, beta)

        # Measure Correlations
        # We measure <O(0) O(t)>
        # Here O(t) is the average "radius" (action density) at time t.

        corrs = []
        for _ in range(50): # 50 measurements
            key, U_flat, _ = sweep(key, U_flat, beta)
            Pt = measure_plaquettes(U_flat)
            # Connected correlator C(t) = <P(0) P(t)> - <P>^2
            # We just store Pt for now
            corrs.append(Pt)

        corrs = np.array(corrs) # (N_meas, L)

        # Compute C(t)
        # Average over reference time t0? No, just use t=0 vs t=dt
        # C(dt) = < P(t) P(t+dt) >_t - <P>^2

        avg_P = np.mean(corrs)
        C_t = np.zeros(L//2)

        for dt in range(L//2):
            val = 0
            for t in range(L):
                val += np.mean(corrs[:, t] * corrs[:, (t+dt)%L])
            val /= L
            C_t[dt] = val - avg_P**2

        # Extract Mass: C(t) ~ A * (exp(-mt) + exp(-m(L-t)))
        # Simple estimator: m_eff(t) = ln(C(t)/C(t+1))

        if C_t[0] > 1e-10 and C_t[1] > 1e-10:
            m_est = np.log(C_t[0] / C_t[1])
        else:
            m_est = 0.0

        print(f"{beta:<10.1f} | {acc/(Vol*4):<10.2f} | {m_est:<15.4f}")
        results[beta] = m_est
    print("\nDone. Check if Mass Gap decreases as Beta increases (Scaling).")
if __name__ == "__main__":
    run_scaling_analysis()

Beta       | Accept     | Mass Gap (est) 
---------------------------------------------
2.5        | 0.80       | 0.0143         
3.0        | 0.77       | 0.1610         
3.5        | 0.75       | 0.0000         
4.0        | 0.72       | 0.0000         
4.5        | 0.72       | 0.0000         

Done. Check if Mass Gap decreases as Beta increases (Scaling).


In [5]:
import jax
import jax.numpy as jnp
from jax import random, jit, lax
import numpy as np # Added for make_neighbor_table
import time

# --- SU(2) Utils ---
sigma_x = jnp.array([[0, 1], [1, 0]], dtype=jnp.complex64)
sigma_y = jnp.array([[0, -1j], [1j, 0]], dtype=jnp.complex64)
sigma_z = jnp.array([[1, 0], [0, -1]], dtype=jnp.complex64)
sigmas = jnp.stack([sigma_x, sigma_y, sigma_z])

@jit
def exp_map_pauli(alpha):
    """Map R^3 vector alpha to SU(2) element U = exp(i * alpha . sigma)."""
    theta2 = jnp.sum(alpha**2)
    theta = jnp.sqrt(theta2 + 1e-10)
    c = jnp.cos(theta)
    s = jnp.sin(theta) / theta
    alpha_dot_sigma = jnp.einsum('k,kij->ij', alpha, sigmas)
    return c * jnp.eye(2, dtype=jnp.complex64) + 1j * s * alpha_dot_sigma

@jit
def random_su2_near_id(key, epsilon):
    """Generate random SU(2) near identity."""
    alpha = random.normal(key, (3,)) * epsilon
    return exp_map_pauli(alpha)

# --- Lattice Utils ---
L = 4
Dim = 4
Vol = L**Dim
NumLinks = Vol * Dim

# Helper to map 4D (x,y,z,t) index to flat site index
@jit
def flat_idx(x, y, z, t):
    return x + L * (y + L * (z + L * t))

# Helper to map flat site index to 4D (x,y,z,t) index
@jit
def site_coords(idx):
    t = idx // (L*L*L)
    rem_t = idx % (L*L*L)
    z = rem_t // (L*L)
    rem_z = rem_t % (L*L)
    y = rem_z // L
    x = rem_z % L
    return x, y, z, t

# This needs to be done on CPU or in a jitted context if possible.
# For now, let's keep it on CPU for neighbor table generation.
def make_neighbor_table_4d(L, Dim):
    UP = np.zeros((Dim, L**Dim), dtype=int)
    DOWN = np.zeros((Dim, L**Dim), dtype=int)
    for t in range(L):
        for z in range(L):
            for y in range(L):
                for x in range(L):
                    idx = flat_idx(x, y, z, t)
                    # Positive directions
                    UP[0, idx] = flat_idx((x + 1) % L, y, z, t)
                    UP[1, idx] = flat_idx(x, (y + 1) % L, z, t)
                    UP[2, idx] = flat_idx(x, y, (z + 1) % L, t)
                    UP[3, idx] = flat_idx(x, y, z, (t + 1) % L)
                    # Negative directions
                    DOWN[0, idx] = flat_idx((x - 1 + L) % L, y, z, t)
                    DOWN[1, idx] = flat_idx(x, (y - 1 + L) % L, z, t)
                    DOWN[2, idx] = flat_idx(x, x, (z - 1 + L) % L, t)
                    DOWN[3, idx] = flat_idx(x, y, z, (t - 1 + L) % L)
    return jnp.array(UP), jnp.array(DOWN)

UP_table, DOWN_table = make_neighbor_table_4d(L, Dim)

# Lattice shape: (L, L, L, L, Dim, 2, 2)
# But for JAX scan, flat array is often easier, but indexing is harder.
# Let's use (L, L, L, L, Dim, 2, 2)

@jit
def init_lattice(key):
    # Cold start (Identity)
    return jnp.stack([jnp.eye(2, dtype=jnp.complex64) for _ in range(Dim)] * Vol).reshape(L, L, L, L, Dim, 2, 2)

@jit
def get_staples(U, x_c, y_c, z_c, t_c, mu):
    """
    Calculate sum of staples for link U_mu(x_c).
    U is the 4D-indexed lattice (L, L, L, L, Dim, 2, 2)
    """
    staple_sum = jnp.zeros((2, 2), dtype=jnp.complex64)

    current_site_flat = flat_idx(x_c, y_c, z_c, t_c)

    for nu in range(Dim):
        # Skip if nu == mu, but JAX needs conditional logic
        # using where or masked operations.
        mask = jnp.where(nu == mu, 0.0, 1.0)

        # Indices for forward staple
        x_plus_mu_coords = site_coords(UP_table[mu, current_site_flat])
        x_plus_nu_coords = site_coords(UP_table[nu, current_site_flat])

        # U_nu(x+mu) U_mu^dag(x+nu) U_nu^dag(x)
        u1 = U[x_plus_mu_coords[0], x_plus_mu_coords[1], x_plus_mu_coords[2], x_plus_mu_coords[3], nu]
        u2 = jnp.conjugate(jnp.transpose(U[x_plus_nu_coords[0], x_plus_nu_coords[1], x_plus_nu_coords[2], x_plus_nu_coords[3], mu]))
        u3 = jnp.conjugate(jnp.transpose(U[x_c, y_c, z_c, t_c, nu]))
        term_fwd = u1 @ u2 @ u3

        # Indices for backward staple
        x_minus_nu_coords = site_coords(DOWN_table[nu, current_site_flat])
        x_plus_mu_minus_nu_coords = site_coords(DOWN_table[nu, UP_table[mu, current_site_flat]])

        # U_nu^dag(x+mu-nu) U_mu^dag(x-nu) U_nu(x-nu)
        u1_b = jnp.conjugate(jnp.transpose(U[x_plus_mu_minus_nu_coords[0], x_plus_mu_minus_nu_coords[1], x_plus_mu_minus_nu_coords[2], x_plus_mu_minus_nu_coords[3], nu]))
        u2_b = jnp.conjugate(jnp.transpose(U[x_minus_nu_coords[0], x_minus_nu_coords[1], x_minus_nu_coords[2], x_minus_nu_coords[3], mu]))
        u3_b = U[x_minus_nu_coords[0], x_minus_nu_coords[1], x_minus_nu_coords[2], x_minus_nu_coords[3], nu]
        term_bwd = u3_b @ u2_b @ u1_b

        staple_sum += mask * (term_fwd + term_bwd)

    return staple_sum

@jit
def update_link(key, U, x, y, z, t, mu, beta):
    """Metropolis update for a single link."""
    U_old = U[x, y, z, t, mu]
    staples = get_staples(U, x, y, z, t, mu)

    # Propose new link
    key, subkey = random.split(key)
    epsilon = 0.2
    delta_U = random_su2_near_id(subkey, epsilon)
    U_new = delta_U @ U_old

    # Calculate action change
    # S_loc = - beta/2 * ReTr(U * Staples^dag)
    # We need to take the dagger of staples, not transpose
    staples_dagger = jnp.conjugate(jnp.transpose(staples))
    tr_old = jnp.real(jnp.trace(U_old @ staples_dagger))
    tr_new = jnp.real(jnp.trace(U_new @ staples_dagger))

    dS = -0.5 * beta * (tr_new - tr_old)

    # Metropolis accept/reject
    key, subkey = random.split(key)
    accept = jnp.log(random.uniform(subkey)) < -dS

    U_final = jnp.where(accept, U_new, U_old)

    # Update lattice
    U = U.at[x, y, z, t, mu].set(U_final)

    return key, U, jnp.where(accept, 1.0, 0.0)

def sweep(key, U, beta):
    """Perform one full sweep over the lattice."""
    acc_count = 0
    # Sequential update (Python loop, will be slow without JAX scan/vmap)
    for x in range(L):
        for y in range(L):
            for z in range(L):
                for t in range(L):
                    for mu in range(Dim):
                        key, U, acc = update_link(key, U, x, y, z, t, mu, beta)
                        acc_count += acc
    return key, U, acc_count / NumLinks

@jit
def measure_defects(U, r_crit):
    """Count links with radius > r_crit."""
    # Radius = arccos(0.5 * ReTr(U))
    # U is (L, L, L, L, Dim, 2, 2)
    tr = jnp.real(jnp.trace(U, axis1=-2, axis2=-1))
    tr = jnp.clip(tr, -2.0, 2.0)
    theta = jnp.arccos(0.5 * tr)

    defects = jnp.sum(theta > r_crit)
    return defects / NumLinks

def run_mc_validation():
    print(f"Starting 4D Lattice MC Validation (L={L}^4)...")
    print("Goal: Measure Defect Density rho(beta) to validate Balaban Scaling.")

    # Scan parameters
    betas = [2.0, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0]
    # Critical radius for convexity (approximate, from Haar bound)
    # r_c ~ pi/2 for strict convexity, but we use the "Large Field" threshold.
    # Let's use a fixed threshold r_c = 1.0 for now to check scaling.
    r_crit = 1.0

    print(f"{'Beta':<10} | {'Accept':<10} | {'Defect Density':<20}")
    print("-" * 50)

    key = random.PRNGKey(42)
    U = init_lattice(key)

    for beta in betas:
        # Thermalize
        for _ in range(50): # Short thermalization for demo
            key, U, acc = sweep(key, U, beta)

        # Measure
        rho_sum = 0.0
        N_meas = 20
        for _ in range(N_meas):
            key, U, acc = sweep(key, U, beta)
            rho = measure_defects(U, r_crit)
            rho_sum += rho

        rho_avg = rho_sum / N_meas
        print(f"{beta:<10.1f} | {acc:<10.2f} | {rho_avg:<20.6f}")

if __name__ == "__main__":
    run_mc_validation()


Starting 4D Lattice MC Validation (L=4^4)...
Goal: Measure Defect Density rho(beta) to validate Balaban Scaling.
Beta       | Accept     | Defect Density      
--------------------------------------------------
2.0        | 0.77       | 0.779883            
3.0        | 0.70       | 0.828272            
4.0        | 0.66       | 0.844482            
5.0        | 0.62       | 0.858008            
6.0        | 0.61       | 0.857959            
8.0        | 0.55       | 0.879150            
10.0       | 0.51       | 0.873242            


In [7]:
import jax
import jax.numpy as jnp
from jax import random, jit, lax
import time

# --- SU(2) Utils ---
sigma_x = jnp.array([[0, 1], [1, 0]], dtype=jnp.complex64)
sigma_y = jnp.array([[0, -1j], [1j, 0]], dtype=jnp.complex64)
sigma_z = jnp.array([[1, 0], [0, -1]], dtype=jnp.complex64)
sigmas = jnp.stack([sigma_x, sigma_y, sigma_z])

@jit
def exp_map_pauli(alpha):
    """Map R^3 vector alpha to SU(2) element U = exp(i * alpha . sigma)."""
    theta2 = jnp.sum(alpha**2)
    theta = jnp.sqrt(theta2 + 1e-10)
    c = jnp.cos(theta)
    s = jnp.sin(theta) / theta
    alpha_dot_sigma = jnp.einsum('k,kij->ij', alpha, sigmas)
    return c * jnp.eye(2, dtype=jnp.complex64) + 1j * s * alpha_dot_sigma

@jit
def random_su2_near_id(key, epsilon):
    """Generate random SU(2) near identity."""
    alpha = random.normal(key, (3,)) * epsilon
    return exp_map_pauli(alpha)

# --- Lattice Utils ---
L = 4
Dim = 4
Vol = L**Dim
NumLinks = Vol * Dim

# Lattice shape: (L, L, L, L, Dim, 2, 2)
# But for JAX scan, flat array is often easier, but indexing is harder.
# Let's use (L, L, L, L, Dim, 2, 2)

@jit
def init_lattice(key):
    # Cold start (Identity)
    return jnp.stack([jnp.eye(2, dtype=jnp.complex64) for _ in range(Dim)] * Vol).reshape(L, L, L, L, Dim, 2, 2)

@jit
def get_staples(U, x, y, z, t, mu):
    """
    Calculate sum of staples for link U_mu(x).
    Staple_nu_up = U_nu(x+mu) U_mu^dag(x+nu) U_nu^dag(x)
    Staple_nu_down = U_nu^dag(x+mu-nu) U_mu^dag(x-nu) U_nu(x-nu)
    """
    staple_sum = jnp.zeros((2, 2), dtype=jnp.complex64)

    # Current site vector
    pos = jnp.array([x, y, z, t])
    U_old = U[x, y, z, t, mu]
    staples = get_staples(U, x, y, z, t, mu)

    # Propose new link
    key, subkey = random.split(key)
    epsilon = 0.2
    delta_U = random_su2_near_id(subkey, epsilon)
    U_new = delta_U @ U_old

    # Calculate action change
    # S_loc = - beta/2 * ReTr(U * Staples_dag)
    # But wait, Staples sum is U_nu(x+mu)... which is the "rest of the plaquette".
    # So U * Staples is the plaquette.
    # We want to maximize ReTr(Plaquette).
    # So we want to maximize ReTr(U * Staples).
    # My previous code used staples_dag, which maximized ReTr(U * Staples_dag).
    # This was wrong. It drove U to be close to Staples (anti-ordering?).

    tr_old = jnp.real(jnp.trace(U_old @ staples))
    tr_new = jnp.real(jnp.trace(U_new @ staples))

    dS = -0.5 * beta * (tr_new - tr_old)

    # Metropolis accept/reject
    key, subkey = random.split(key)
    accept = jnp.log(random.uniform(subkey)) < -dS

    U_final = jnp.where(accept, U_new, U_old)

    # Update lattice in place? JAX arrays are immutable.
    # We must return the new lattice. This is slow in pure Python loop,
    # but we can't easily vmap over dependent updates.
    # For small lattice, we can use index_update.
    U = U.at[x, y, z, t, mu].set(U_final)

    return key, U, jnp.where(accept, 1.0, 0.0)

def sweep(key, U, beta):
    """Perform one full sweep over the lattice."""
    acc_count = 0
    # Sequential update
    for x in range(L):
        for y in range(L):
            for z in range(L):
                for t in range(L):
                    for mu in range(Dim):
                        key, U, acc = update_link(key, U, x, y, z, t, mu, beta)
                        acc_count += acc
    return key, U, acc_count / NumLinks

@jit
def measure_defects(U, r_crit):
    """Count links with radius > r_crit."""
    # Radius = arccos(0.5 * ReTr(U))
    tr = jnp.real(jnp.trace(U, axis1=-2, axis2=-1))
    tr = jnp.clip(tr, -2.0, 2.0)
    theta = jnp.arccos(0.5 * tr)

    defects = jnp.sum(theta > r_crit)
    return defects / NumLinks

def run_mc_validation():
    print(f"Starting 4D Lattice MC Validation (L={L}^4)...")
    print("Goal: Measure Defect Density rho(beta) to validate Balaban Scaling.")

    # Scan parameters
    betas = [2.0, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0]
    # Critical radius for convexity (approximate, from Haar bound)
    # r_c ~ pi/2 for strict convexity, but we use the "Large Field" threshold.
    # Let's use a fixed threshold r_c = 1.0 for now to check scaling.
    r_crit = 1.0

    print(f"{'Beta':<10} | {'Accept':<10} | {'Defect Density':<20}")
    print("-" * 50)

    key = random.PRNGKey(42)
    U = init_lattice(key)

    for beta in betas:
        # Thermalize
        for _ in range(50): # Short thermalization for demo
            key, U, acc = sweep(key, U, beta)

        # Measure
        rho_sum = 0.0
        N_meas = 20
        for _ in range(N_meas):
            key, U, acc = sweep(key, U, beta)
            rho = measure_defects(U, r_crit)
            rho_sum += rho

        rho_avg = rho_sum / N_meas
        print(f"{beta:<10.1f} | {acc:<10.2f} | {rho_avg:<20.6f}")

if __name__ == "__main__":
    run_mc_validation()

Starting 4D Lattice MC Validation (L=4^4)...
Goal: Measure Defect Density rho(beta) to validate Balaban Scaling.
Beta       | Accept     | Defect Density      
--------------------------------------------------
2.0        | 0.77       | 0.779883            
3.0        | 0.70       | 0.828272            
4.0        | 0.66       | 0.844482            
5.0        | 0.62       | 0.858008            
6.0        | 0.61       | 0.857959            
8.0        | 0.55       | 0.879150            
10.0       | 0.51       | 0.873242            


In [11]:
import numpy as np

# ============================================
# Minimal SU(2) + Lattice + Metropolis — PRINTS IMMEDIATELY
# ============================================

def random_su2(eps=0.3):
    alpha = np.random.normal(0, eps, 3)
    a = np.linalg.norm(alpha)
    if a < 1e-12:
        return np.eye(2)
    nx = alpha / a
    c, s = np.cos(a), np.sin(a)
    sigma = np.array([
        [[0,1],[1,0]],
        [[0,-1j],[1j,0]],
        [[1,0],[0,-1]]
    ], dtype=np.complex128)
    M = nx[0]*sigma[0] + nx[1]*sigma[1] + nx[2]*sigma[2]
    return c*np.eye(2) + 1j*s*M

def su2_dagger(U): return U.conjugate().T

# Lattice: 2D for testing
L = 4
dim = 2

# Initialize lattice
U = np.zeros((L, L, dim, 2, 2), dtype=np.complex128)
for x in range(L):
    for y in range(L):
        for mu in range(dim):
            U[x,y,mu] = np.eye(2)

def shift2(x, y, mu, direction=+1):
    if mu == 0: return ((x + direction) % L, y)
    if mu == 1: return (x, (y + direction) % L)

def staple(U, x, y, mu, beta):
    S = np.zeros((2,2), dtype=np.complex128)
    for nu in range(dim):
        if nu == mu: continue
        # forward
        xn, yn = shift2(x, y, nu, +1)
        xm, ym = shift2(x, y, mu, +1)
        xmnu, ymnu = shift2(xm, ym, nu, +1)
        S += U[x,y,nu] @ U[xn,yn,mu] @ su2_dagger(U[xm,ym,nu])
        # backward
        xm1, ym1 = shift2(x, y, nu, -1)
        xm2, ym2 = shift2(xm, ym, nu, -1)
        S += su2_dagger(U[xm1,ym1,nu]) @ su2_dagger(U[xm1,ym1,mu]) @ U[xm2,ym2,nu]
    return S

def sweep(U, beta=2.5):
    accepts = 0
    for x in range(L):
        for y in range(L):
            for mu in range(dim):
                U_old = U[x,y,mu]
                S = staple(U, x, y, mu, beta)
                U_new = random_su2(0.25) @ U_old
                dS = -0.5*beta*(np.real(np.trace(U_new @ S)) -
                               np.real(np.trace(U_old @ S)))
                if np.random.rand() < np.exp(-dS):
                    U[x,y,mu] = U_new
                    accepts += 1
    return accepts

def measure_plaquette(U):
    total, count = 0, 0
    for x in range(L):
        for y in range(L):
            xn, yn = shift2(x, y, 0, +1)
            xm, ym = shift2(x, y, 1, +1)
            P = U[x,y,0] @ U[xn,yn,1] @ su2_dagger(U[xm,ym,0]) @ su2_dagger(U[x,y,1])
            total += np.real(np.trace(P))
            count += 1
    return total / count

# RUN TEST
print("Running minimal 2D SU(2) test...")
for i in range(10):
    acc = sweep(U, beta=2.5)
    plaq = measure_plaquette(U)
    print(f"Sweep {i}: accept={acc}, avg_plaquette={plaq:.4f}")


Running minimal 2D SU(2) test...
Sweep 0: accept=26, avg_plaquette=1.5993
Sweep 1: accept=21, avg_plaquette=1.2992
Sweep 2: accept=24, avg_plaquette=0.9619
Sweep 3: accept=28, avg_plaquette=0.9851
Sweep 4: accept=23, avg_plaquette=1.0190
Sweep 5: accept=23, avg_plaquette=0.9823
Sweep 6: accept=26, avg_plaquette=0.9718
Sweep 7: accept=30, avg_plaquette=0.6079
Sweep 8: accept=24, avg_plaquette=0.2131
Sweep 9: accept=22, avg_plaquette=0.2889


In [12]:
import jax
import jax.numpy as jnp
from jax import random, jit, vmap, lax
import time

# --- CONFIGURATION ---
jax.config.update("jax_enable_x64", True) # Precision for small defects

# --- SU(2) ALGEBRA ---
sigma_x = jnp.array([[0, 1], [1, 0]], dtype=jnp.complex128)
sigma_y = jnp.array([[0, -1j], [1j, 0]], dtype=jnp.complex128)
sigma_z = jnp.array([[1, 0], [0, -1]], dtype=jnp.complex128)
sigmas = jnp.stack([sigma_x, sigma_y, sigma_z])

@jit
def exp_map_pauli(alpha):
    """Map R^3 vector alpha to SU(2) element U = exp(i * alpha . sigma)."""
    theta2 = jnp.sum(alpha**2)
    theta = jnp.sqrt(theta2 + 1e-12)
    c = jnp.cos(theta)
    s = jnp.sin(theta) / theta
    alpha_dot_sigma = jnp.einsum('k,kij->ij', alpha, sigmas)
    return c * jnp.eye(2, dtype=jnp.complex128) + 1j * s * alpha_dot_sigma

@jit
def random_su2_near_id(key, epsilon):
    """Generate random SU(2) near identity."""
    alpha = random.normal(key, (3,)) * epsilon
    return exp_map_pauli(alpha)

# --- LATTICE GEOMETRY ---
L = 4
Dim = 4
Vol = L**Dim
NumLinks = Vol * Dim

# Lattice State: (L, L, L, L, Dim, 2, 2)
# We use a flat array for JAX scan efficiency if needed, but here we stick to
# the 5D tensor for clarity in indexing.

@jit
def init_lattice(key):
    # Cold start (Identity)
    return jnp.stack([jnp.eye(2, dtype=jnp.complex128) for _ in range(Dim)] * Vol).reshape(L, L, L, L, Dim, 2, 2)

@jit
def get_staples(U, x, y, z, t, mu):
    """
    Calculate sum of staples for link U_mu(x).
    Staple is the sum of the product of the OTHER 3 links in each of the 6 plaquettes sharing U_mu(x).

    For a plaquette U_mu(x) U_nu(x+mu) U_mu^dag(x+nu) U_nu^dag(x),
    the "staple" for U_mu(x) is U_nu(x+mu) U_mu^dag(x+nu) U_nu^dag(x).
    Wait, no.
    S_plaq = ReTr(U_mu(x) * U_nu(x+mu) * U_mu(x+nu)^dag * U_nu(x)^dag)
    To maximize this w.r.t U_mu(x), we write it as ReTr(U_mu(x) * A).
    Then A = U_nu(x+mu) * U_mu(x+nu)^dag * U_nu(x)^dag.

    So "Staple" A is indeed the product of the other 3 links.
    """
    staple_sum = jnp.zeros((2, 2), dtype=jnp.complex128)
    pos = jnp.array([x, y, z, t])
    dirs = jnp.eye(4, dtype=jnp.int32)
    mu_vec = dirs[mu]

    # Unrolled loop over nu != mu
    for nu in range(4):
        if nu == mu: continue
        nu_vec = dirs[nu]

        # Forward Plaquette: x -> x+mu -> x+mu+nu -> x+nu -> x
        # Links: U_mu(x), U_nu(x+mu), U_mu(x+nu)^dag, U_nu(x)^dag
        # Staple_up = U_nu(x+mu) @ U_mu(x+nu)^dag @ U_nu(x)^dag

        p_x_mu = (pos + mu_vec) % L
        p_x_nu = (pos + nu_vec) % L
        # p_x_mu_nu = (pos + mu_vec + nu_vec) % L # Not needed for indexing

        U_nu_x_mu = U[p_x_mu[0], p_x_mu[1], p_x_mu[2], p_x_mu[3], nu]
        U_mu_x_nu = U[p_x_nu[0], p_x_nu[1], p_x_nu[2], p_x_nu[3], mu]
        U_nu_x = U[x, y, z, t, nu]

        S_up = U_nu_x_mu @ jnp.conjugate(U_mu_x_nu.T) @ jnp.conjugate(U_nu_x.T)
        staple_sum += S_up

        # Backward Plaquette: x -> x+mu -> x+mu-nu -> x-nu -> x
        # Links: U_mu(x), U_nu(x+mu-nu)^dag, U_mu(x-nu)^dag, U_nu(x-nu)
        # Wait, let's trace carefully.
        # Plaquette in (mu, -nu) plane.
        # Path: x -> x+mu -> x+mu-nu -> x-nu -> x
        # U_mu(x) * U_nu(x+mu-nu)^dag * U_mu(x-nu)^dag * U_nu(x-nu)
        # Staple_down = U_nu(x+mu-nu)^dag @ U_mu(x-nu)^dag @ U_nu(x-nu)

        p_x_minus_nu = (pos - nu_vec) % L
        p_x_mu_minus_nu = (pos + mu_vec - nu_vec) % L

        U_nu_x_minus_nu = U[p_x_minus_nu[0], p_x_minus_nu[1], p_x_minus_nu[2], p_x_minus_nu[3], nu]
        U_mu_x_minus_nu = U[p_x_minus_nu[0], p_x_minus_nu[1], p_x_minus_nu[2], p_x_minus_nu[3], mu]
        U_nu_x_mu_minus_nu = U[p_x_mu_minus_nu[0], p_x_mu_minus_nu[1], p_x_mu_minus_nu[2], p_x_mu_minus_nu[3], nu]

        S_down = jnp.conjugate(U_nu_x_mu_minus_nu.T) @ jnp.conjugate(U_mu_x_minus_nu.T) @ U_nu_x_minus_nu
        staple_sum += S_down

    return staple_sum

@jit
def update_link(key, U, x, y, z, t, mu, beta):
    """Metropolis update for a single link U_mu(x)."""
    U_old = U[x, y, z, t, mu]
    staples = get_staples(U, x, y, z, t, mu)

    # Propose new link: U' = R * U
    key, subkey = random.split(key)
    epsilon = 0.2
    R = random_su2_near_id(subkey, epsilon)
    U_new = R @ U_old

    # Action: S = -beta/2 * Sum ReTr(Plaquette)
    # Local part: -beta/2 * ReTr(U * Staples)
    # We want to MINIMIZE S, so MAXIMIZE ReTr(U * Staples)
    # dS = S_new - S_old = -beta/2 * (ReTr(U_new * Staples) - ReTr(U_old * Staples))

    tr_old = jnp.real(jnp.trace(U_old @ staples))
    tr_new = jnp.real(jnp.trace(U_new @ staples))

    dS = -0.5 * beta * (tr_new - tr_old)

    # Metropolis: Accept if rand < exp(-dS)
    # log(rand) < -dS
    # log(rand) < 0.5 * beta * (tr_new - tr_old)

    key, subkey = random.split(key)
    accept_prob = 0.5 * beta * (tr_new - tr_old)
    accept = jnp.log(random.uniform(subkey)) < accept_prob

    U_final = jnp.where(accept, U_new, U_old)
    U = U.at[x, y, z, t, mu].set(U_final)

    return key, U, jnp.where(accept, 1.0, 0.0)

def sweep(key, U, beta):
    """Sequential sweep over the lattice."""
    acc_count = 0
    # We can't vmap the update because it's sequential (detailed balance).
    # But we can jit the whole sweep loop for small L.
    # For L=4, 4^4*4 = 1024 links. Unrolling might be too much for compiler.
    # We keep the python loop but jit the update.
    for x in range(L):
        for y in range(L):
            for z in range(L):
                for t in range(L):
                    for mu in range(Dim):
                        key, U, acc = update_link(key, U, x, y, z, t, mu, beta)
                        acc_count += acc
    return key, U, acc_count / NumLinks

# --- OBSERVABLES ---

@jit
def measure_plaquette_defects(U, r_crit):
    """
    Measure fraction of PLAQUETTES that are 'defects'.
    Defect: Plaquette trace < threshold.
    r_crit is in terms of geodesic distance on SU(2).
    Trace = 2 cos(theta).
    Defect if theta > r_crit.
    """
    # We need to compute all plaquettes.
    # P_mu_nu(x) = U_mu(x) U_nu(x+mu) U_mu(x+nu)^dag U_nu(x)^dag

    defects = 0.0
    total_plaqs = 0.0

    # Loop over all plaquettes (x, mu < nu)
    # We can vmap this!

    # Helper to compute one plaquette trace
    def get_plaq_trace(pos, mu, nu):
        dirs = jnp.eye(4, dtype=jnp.int32)
        mu_vec = dirs[mu]
        nu_vec = dirs[nu]

        p_x = pos
        p_x_mu = (pos + mu_vec) % L
        p_x_nu = (pos + nu_vec) % L

        U_mu_x = U[p_x[0], p_x[1], p_x[2], p_x[3], mu]
        U_nu_x_mu = U[p_x_mu[0], p_x_mu[1], p_x_mu[2], p_x_mu[3], nu]
        U_mu_x_nu = U[p_x_nu[0], p_x_nu[1], p_x_nu[2], p_x_nu[3], mu]
        U_nu_x = U[p_x[0], p_x[1], p_x[2], p_x[3], nu]

        P = U_mu_x @ U_nu_x_mu @ jnp.conjugate(U_mu_x_nu.T) @ jnp.conjugate(U_nu_x.T)
        return jnp.real(jnp.trace(P))

    # Generate all site indices
    sites = jnp.indices((L, L, L, L)).reshape(4, -1).T # (Vol, 4)

    # Compute defects for each plane
    for mu in range(Dim):
        for nu in range(mu + 1, Dim):
            # Vmap over all sites for this plane
            tr_vals = vmap(lambda pos: get_plaq_trace(pos, mu, nu))(sites)

            # Convert to radius: theta = arccos(tr/2)
            tr_vals = jnp.clip(tr_vals, -2.0, 2.0)
            thetas = jnp.arccos(0.5 * tr_vals)

            # Count defects
            defects += jnp.sum(thetas > r_crit)
            total_plaqs += Vol

    return defects / total_plaqs

def run_simulation():
    print(f"Starting CORRECTED 4D Lattice MC (L={L}^4)...")
    print("Goal: Measure PLAQUETTE Defect Density rho(beta).")
    print("Defect Condition: Plaquette Geodesic Radius > 1.9248 (Haar Convexity)")

    betas = [2.0, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0]
    r_crit = 1.9248 # The correct convexity threshold

    print(f"{'Beta':<10} | {'Accept':<10} | {'Plaq Defect Density':<25}")
    print("-" * 55)

    key = random.PRNGKey(42)
    U = init_lattice(key)

    for beta in betas:
        # Thermalize
        # For L=4, 100 sweeps is usually enough for local observables
        for _ in range(100):
            key, U, acc = sweep(key, U, beta)

        # Measure
        rho_sum = 0.0
        N_meas = 20
        for _ in range(N_meas):
            key, U, acc = sweep(key, U, beta) # Decorrelate
            rho = measure_plaquette_defects(U, r_crit)
            rho_sum += rho

        rho_avg = rho_sum / N_meas
        print(f"{beta:<10.1f} | {acc:<10.2f} | {rho_avg:<25.6e}")

if __name__ == "__main__":
    run_simulation()


Starting CORRECTED 4D Lattice MC (L=4^4)...
Goal: Measure PLAQUETTE Defect Density rho(beta).
Defect Condition: Plaquette Geodesic Radius > 1.9248 (Haar Convexity)
Beta       | Accept     | Plaq Defect Density      
-------------------------------------------------------


AttributeError: module 'jax.random' has no attribute 'PRPRNGKey'